In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go


In [ ]:
df_hdi = pd.read_excel('https://raw.githubusercontent.com/izzzav/hse_visualization/refs/heads/main/hdr-data.xlsx')
df_jobs = pd.read_csv('https://raw.githubusercontent.com/izzzav/hse_visualization/refs/heads/main/JobsCSV.csv')


In [25]:
def get_df_from_year(year, df_local):
    cols_to_select = ['Country Code', 'Indicator Code', year]
    cols_to_select.insert(0, 'Country Name')
    df_filtered = df_local[cols_to_select].copy()
    df_pivoted = df_filtered.pivot(
        index=['Country Name', 'Country Code'],
        columns='Indicator Code', 
        values=year
    ).reset_index()
    return df_pivoted
df = get_df_from_year('2016', df_jobs)
df

Indicator Code,Country Name,Country Code,BM.KLT.DINV.WD.GD.ZS,BM.TRF.PWKR.CD.DT,BX.KLT.DINV.WD.GD.ZS,BX.TRF.PWKR.CD,CM.MKT.LCAP.GD.ZS,EG.ELC.ACCS.ZS,EG.USE.ELEC.KH.PC,EN.POP.DNST,...,SP.POP.TOTL,SP.RUR.TOTL,SP.RUR.TOTL.ZS,SP.URB.TOTL,SP.URB.TOTL.IN.ZS,TM.VAL.ICTG.ZS.UN,TX.QTY.MRCH.XD.WD,TX.VAL.FUEL.ZS.UN,TX.VAL.MRCH.XD.WD,TX.VAL.TECH.MF.ZS
0,Afghanistan,AFG,-0.075895,1.182164e+08,0.480710,2.774835e+08,NaN,84.137138,NaN,53.083405,...,3.465603e+07,2.598509e+07,74.980000,8.670939e+06,25.020000,0.262056,185.045514,NaN,434.379370,NaN
1,Albania,ALB,0.059209,1.471281e+08,8.786734,9.960014e+08,NaN,100.000000,NaN,104.967190,...,2.876101e+06,1.195854e+06,41.579000,1.680247e+06,58.421000,2.941437,556.660364,11.178982,750.385083,0.647751
2,Algeria,DZA,0.029423,5.897023e+07,1.029475,1.696801e+07,NaN,99.439568,NaN,17.048895,...,4.060605e+07,1.158937e+07,28.541000,2.901668e+07,71.459000,5.181022,144.936023,93.992128,131.099918,0.343524
3,American Samoa,ASM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,277.995000,...,5.559900e+04,7.118000e+03,12.802000,4.848100e+04,87.198000,NaN,44.999741,NaN,112.619116,NaN
4,Andorra,AND,NaN,NaN,NaN,NaN,NaN,100.000000,NaN,164.427660,...,7.728100e+04,9.082000e+03,11.752000,6.819900e+04,88.248000,NaN,149.884060,NaN,186.046284,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237,West Bank and Gaza,PSE,-0.335063,3.189228e+07,2.208254,2.010197e+08,25.250974,100.000000,NaN,756.074086,...,4.551566e+06,1.109308e+06,24.372000,3.442258e+06,75.628000,3.103988,132.926211,0.163516,199.093781,0.593923
238,World,WLD,2.660863,3.942136e+11,3.119152,3.437027e+11,97.783324,87.378443,NaN,57.380523,...,7.444157e+09,3.395702e+09,45.626997,4.046607e+09,54.373003,12.990466,NaN,11.346811,NaN,17.883797
239,"Yemen, Rep.",YEM,0.191607,NaN,-3.080162,NaN,NaN,71.642349,NaN,52.245796,...,2.758421e+07,1.782106e+07,64.606000,9.763156e+06,35.394000,NaN,11.129887,NaN,24.515813,NaN
240,Zambia,ZMB,0.843340,6.340584e+07,3.163072,3.846444e+07,NaN,27.219337,NaN,22.318554,...,1.659139e+07,9.550336e+06,57.562000,7.041054e+06,42.438000,NaN,267.841100,NaN,650.078203,NaN


In [26]:
df = get_df_from_year('1990', df_jobs)
display(df)
map_data = df[['Country Name', 'Country Code', 'EG.ELC.ACCS.ZS', 'SP.POP.TOTL']].copy()
map_data.columns = ['Country', 'Country_Code', 'Electricity Access', 'Population']
map_data['Population_without_electricity'] = map_data['Population'] * (1 - map_data['Electricity Access'] / 100)
map_data = map_data.dropna(subset=['Electricity Access'])
display(map_data)
fig = px.choropleth(
    map_data,
    locations='Country_Code',
    locationmode='ISO-3',
    color='Electricity Access',
    hover_name='Country',
    hover_data={'Electricity Access': ':.2f%', 'Country_Code': False, 'Population': ':,.0f', 'Population_without_electricity': ':,.0f'},
    color_continuous_scale='Viridis',
    range_color=(0, 100),
    title='Access to Electricity in 2010 (% of Population)',
    labels={'Electricity Access': 'Access Rate (%)'},
    projection='natural earth'
)
fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth',
        bgcolor='rgba(0,0,0,0)'
    ),
    margin={"r": 0, "t": 70, "l": 0, "b": 0},
    height=450,
    title_font_size=16
)
fig.update_coloraxes(
    colorbar_title_text='Access Rate (%)',
    colorbar_title_side='right'
)
fig.update_traces(
    hovertemplate='<b style="color:#FFFFFF; font-size:16px">%{hovertext}</b><br>' +
                  '<span style="color:#CCCCCC">Population:</span> <span style="color:#FFFFFF; font-weight:bold">%{customdata[1]:,.0f}</span><br>' +
                  '<span style="color:#CCCCCC">Population without electricity:</span> <span style="color:#FFFFFF; font-weight:bold">%{customdata[2]:,.0f}</span><br>' +
                  '<span style="color:#CCCCCC">Access:</span> <span style="color:#FFFFFF; font-weight:bold">%{customdata[0]:.2f}%</span><extra></extra>',
    customdata=map_data[['Electricity Access', 'Population', 'Population_without_electricity']].values
)
fig.write_html('electricity_map.html')
fig.show(renderer='jupyterlab')

Indicator Code,Country Name,Country Code,BM.KLT.DINV.WD.GD.ZS,BM.TRF.PWKR.CD.DT,BX.KLT.DINV.WD.GD.ZS,BX.TRF.PWKR.CD,CM.MKT.LCAP.GD.ZS,EG.ELC.ACCS.ZS,EG.USE.ELEC.KH.PC,EN.POP.DNST,...,SP.POP.TOTL,SP.RUR.TOTL,SP.RUR.TOTL.ZS,SP.URB.TOTL,SP.URB.TOTL.IN.ZS,TM.VAL.ICTG.ZS.UN,TX.QTY.MRCH.XD.WD,TX.VAL.FUEL.ZS.UN,TX.VAL.MRCH.XD.WD,TX.VAL.TECH.MF.ZS
0,Afghanistan,AFG,NaN,NaN,NaN,NaN,NaN,0.010000,NaN,18.762237,...,1.224911e+07,9.655119e+06,78.823000,2.593995e+06,21.177000,NaN,NaN,NaN,127.081081,NaN
1,Albania,ALB,NaN,NaN,NaN,0.000000e+00,NaN,100.000000,552.252185,119.946788,...,3.286542e+06,2.089320e+06,63.572000,1.197222e+06,36.428000,NaN,NaN,NaN,NaN,NaN
2,Algeria,DZA,0.007557,3.148197e+07,0.000540,3.524418e+08,NaN,98.271378,528.434936,10.879595,...,2.591237e+07,1.241591e+07,47.915000,1.349646e+07,52.085000,NaN,77.193995,96.473992,67.930716,NaN
3,American Samoa,ASM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,235.190000,...,4.703800e+04,8.962000e+03,19.052000,3.807600e+04,80.948000,NaN,NaN,NaN,89.884393,NaN
4,Andorra,AND,NaN,NaN,NaN,NaN,NaN,100.000000,NaN,115.976596,...,5.450900e+04,2.882000e+03,5.288000,5.162700e+04,94.712000,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237,West Bank and Gaza,PSE,NaN,NaN,NaN,NaN,NaN,97.354027,NaN,328.612625,...,1.978248e+06,6.387960e+05,32.291000,1.339452e+06,67.709000,NaN,NaN,NaN,NaN,NaN
238,World,WLD,1.304940,6.524823e+10,0.907785,NaN,53.031899,71.390577,2124.770523,40.766514,...,5.288103e+09,3.010745e+09,56.954679,2.275467e+09,43.045321,NaN,NaN,10.670180,NaN,18.181408
239,"Yemen, Rep.",YEM,NaN,1.064000e+08,-2.317942,NaN,NaN,36.009098,122.003421,22.836599,...,1.205704e+07,9.533380e+06,79.069000,2.523659e+06,20.931000,NaN,NaN,NaN,16.965982,NaN
240,Zambia,ZMB,NaN,1.490000e+07,6.170064,0.000000e+00,NaN,13.900000,762.776506,10.798172,...,8.027253e+06,4.863953e+06,60.593000,3.163300e+06,39.407000,NaN,93.178058,NaN,196.605703,NaN


,Country,Country_Code,Electricity Access,Population,Population_without_electricity
0,Afghanistan,AFG,0.010000,1.224911e+07,1.224789e+07
1,Albania,ALB,100.000000,3.286542e+06,0.000000e+00
2,Algeria,DZA,98.271378,2.591237e+07,4.479270e+05
4,Andorra,AND,100.000000,5.450900e+04,0.000000e+00
5,Angola,AGO,11.397808,1.217144e+07,1.078416e+07
...,...,...,...,...,...
237,West Bank and Gaza,PSE,97.354027,1.978248e+06,5.234391e+04
238,World,WLD,71.390577,5.288103e+09,1.512896e+09
239,"Yemen, Rep.",YEM,36.009098,1.205704e+07,7.715408e+06
240,Zambia,ZMB,13.900000,8.027253e+06,6.911465e+06


In [27]:
exclude_patterns = ['World', 'East Asia & Pacific', 'Arab World', 'Euro area', 'Europe & Central Asia', 'European Union', 'Heavily Indebted', 'income', 'Latin America', 'developed', 'Middle East', 'member', 'North America', 'South Asia', 'Sub-Saharan']
map_data_filtered = map_data[
    ~map_data['Country'].str.contains('|'.join(exclude_patterns), case=False, na=False)
].copy()
map_data_filtered['Population_with_electricity'] = map_data_filtered['Population'] * (map_data_filtered['Electricity Access'] / 100)
total_with_electricity = map_data_filtered['Population_with_electricity'].sum()
total_without_electricity = map_data_filtered['Population_without_electricity'].sum()
total_population = map_data_filtered['Population'].sum()
pie_data = pd.DataFrame({
    'Category': ['With Electricity', 'Without Electricity'],
    'Population': [total_with_electricity, total_without_electricity]
})
fig_pie = px.pie(
    pie_data,
    values='Population',
    names='Category',
    title='Global Population: Access to Electricity (1990)',
    color_discrete_map={
        'With Electricity': '#2E86AB',
        'Without Electricity': '#F18F01'
    }
)
fig_pie.update_traces(
    textposition='inside',
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>' +
                  'Population: %{value:,.0f}<br>' +
                  'Percentage: %{percent}<extra></extra>'
)
fig_pie.update_layout(
    font_size=14,
    title_font_size=18,
    height=600,
    showlegend=True
)
fig_pie.show(renderer='jupyterlab')
print(f"Total Population: {total_population:,.0f}")
print(f"With Electricity: {total_with_electricity:,.0f} ({total_with_electricity/total_population*100:.2f}%)")
print(f"Without Electricity: {total_without_electricity:,.0f} ({total_without_electricity/total_population*100:.2f}%)")

Total Population: 5,201,808,498
With Electricity: 3,713,601,100 (71.39%)
Without Electricity: 1,488,207,398 (28.61%)


In [28]:
import plotly.graph_objects as go
df = get_df_from_year('2010', df_jobs)
corr_data = df[['Country Name']].copy()
corr_data['Literacy Rate'] = df['SE.ADT.LITR.ZS']
corr_data['Tax Revenue (% of GDP)'] = df['GC.TAX.TOTL.GD.ZS']
corr_data['GDP per capita'] = df['NY.GDP.PCAP.KD']
corr_data['Urban Population (%)'] = df['SP.URB.TOTL.IN.ZS']
exclude_patterns = ['&', 'excluding', 'income', 'World', 'Union', 'area', 'members']
corr_data = corr_data[
    ~corr_data['Country Name'].str.contains('|'.join(exclude_patterns), case=False, na=False)
].copy()
corr_data_clean = corr_data[['Literacy Rate', 'Tax Revenue (% of GDP)', 'GDP per capita', 'Urban Population (%)']].dropna()
correlation_matrix = corr_data_clean.corr()
fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale=[[0, '#2166ac'], [0.5, '#f7f7f7'], [1, '#b2182b']],
    zmid=0,
    zmin=-1,
    zmax=1,
    text=correlation_matrix.values.round(2),
    texttemplate='%{text}',
    textfont={"size": 14, "color": "black"},
    colorbar=dict(
        title="Correlation",
        tickmode="linear",
        tick0=-1,
        dtick=0.5
    )
))
fig.update_layout(
    title='Correlation Matrix: Literacy, Taxes, GDP, Urban Population (2010)',
    width=700,
    height=650,
    xaxis={'side': 'bottom', 'tickangle': -45},
    yaxis={'autorange': 'reversed'},
    margin=dict(l=120, r=50, t=80, b=120)
)
fig.show(renderer='jupyterlab')
print(f"Number of countries analyzed: {len(corr_data_clean)}")

Number of countries analyzed: 40


In [29]:
brics_countries = ['Brazil', 'Russian Federation', 'India', 'China', 'South Africa']
post_soviet_countries = [
    'Armenia', 'Azerbaijan', 'Belarus', 'Estonia', 'Georgia', 
    'Kazakhstan', 'Kyrgyzstan', 'Latvia', 'Lithuania', 'Moldova', 
    'Tajikistan', 'Turkmenistan', 'Ukraine', 'Uzbekistan'
]
target_countries = brics_countries + post_soviet_countries
east_countries = df_hdi[df_hdi['country'].isin(target_countries)]
hdi_data = east_countries[
    (east_countries['indicator'] == 'Human Development Index (value)')
].copy()
print(f"\nFound {len(hdi_data)} HDI records")
print(f"Countries: {hdi_data['country'].unique()}")
print(f"Year range: {hdi_data['year'].min()} - {hdi_data['year'].max()}")
hdi_col = 'value' if 'value' in hdi_data.columns else 'actualValue'
hdi_clean = hdi_data.dropna(subset=[hdi_col, 'year', 'country']).copy()
hdi_plot = hdi_clean.groupby(['country', 'year'])[hdi_col].mean().reset_index()
fig = px.line(
    hdi_plot,
    x='year',
    y=hdi_col,
    color='country',
    title='Human Development Index (HDI) Development: BRICS + Post-Soviet Countries',
    labels={hdi_col: 'HDI Value', 'year': 'Year', 'country': 'Country'},
    markers=False,
    line_shape='linear'
)
fig.update_layout(
    width=1200,
    height=700,
    title_font_size=18,
    xaxis_title='Year',
    yaxis_title='HDI Value',
    legend=dict(
        title='Country',
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.01,
        font=dict(size=10)
    ),
    hovermode='closest'
)
brics_colors = px.colors.qualitative.Set1[:5]
post_soviet_colors = px.colors.qualitative.Set2[:14]
for i, trace in enumerate(fig.data):
    country_name = trace.name
    if country_name in brics_countries:
        color_idx = brics_countries.index(country_name) % len(brics_colors)
        fig.data[i].line.color = brics_colors[color_idx]
        fig.data[i].line.width = 3
    elif country_name in post_soviet_countries:
        color_idx = post_soviet_countries.index(country_name) % len(post_soviet_colors)
        fig.data[i].line.color = post_soviet_colors[color_idx]
        fig.data[i].line.width = 2
    fig.data[i].update(
        hovertemplate='<b>%{fullData.name}</b><br>Year: %{x}<br>HDI: %{y:.3f}<extra></extra>',
        hoverlabel=dict(
            bgcolor='white',
            bordercolor=trace.line.color,
            font_size=12,
            font_family="Arial"
        ),
        marker=dict(size=8)
    )
fig.update_layout(
    template='plotly_white',
    hoverdistance=100
)
fig.update_layout(
    hoverlabel=dict(
        bgcolor="white",
        font_size=12,
        font_family="Arial"
    )
)
fig.show()
print("\n=== Summary Statistics ===")
summary = hdi_plot.groupby('country')[hdi_col].agg(['min', 'max', 'mean', 'last']).round(3)
summary.columns = ['Min HDI', 'Max HDI', 'Mean HDI', 'Latest HDI']
summary = summary.sort_values('Latest HDI', ascending=False)
display(summary)


Found 564 HDI records
Countries: ['Armenia' 'Azerbaijan' 'Belarus' 'Brazil' 'China' 'Estonia' 'Georgia'
 'India' 'Kazakhstan' 'Kyrgyzstan' 'Lithuania' 'Latvia'
 'Russian Federation' 'Tajikistan' 'Turkmenistan' 'Ukraine' 'Uzbekistan'
 'South Africa']
Year range: 1990 - 2023



=== Summary Statistics ===


,Min HDI,Max HDI,Mean HDI,Latest HDI
country,,,,
Estonia,0.711,0.905,0.830,0.905
Lithuania,0.699,0.895,0.817,0.895
Latvia,0.686,0.889,0.804,0.889
Georgia,0.705,0.844,0.780,0.844
Kazakhstan,0.665,0.837,0.753,0.837
Russian Federation,0.712,0.847,0.787,0.832
Belarus,0.692,0.828,0.780,0.824
Armenia,0.618,0.811,0.717,0.811
China,0.491,0.797,0.662,0.797


In [30]:
gdi_data = df_hdi[df_hdi['index'] == 'Gender Development Index'].copy()
gii_data = df_hdi[df_hdi['index'] == 'Gender Development Index'].copy()


In [31]:
hdi_data = df_hdi[
    (df_hdi['indicator'] == 'Human Development Index (value)')
].copy()
print(f"Найдено {len(hdi_data)} записей HDI")
print(f"Страны: {hdi_data['country'].nunique()}")
print(f"Годы: {hdi_data['year'].min()} - {hdi_data['year'].max()}")
hdi_col = 'value' if 'value' in hdi_data.columns else 'actualValue'
hdi_clean = hdi_data.dropna(subset=[hdi_col, 'year', 'countryIsoCode']).copy()
hdi_map = hdi_clean.groupby(['countryIsoCode', 'country', 'year'])[hdi_col].mean().reset_index()
hdi_map = hdi_map.rename(columns={
    'countryIsoCode': 'Country_Code',
    'country': 'Country',
    hdi_col: 'HDI',
    'year': 'Year'
})
print(f"\nДанные для карты: {len(hdi_map)} записей")
print(f"Уникальных стран: {hdi_map['Country_Code'].nunique()}")
fig = px.choropleth(
    hdi_map,
    locations='Country_Code',
    locationmode='ISO-3',
    color='HDI',
    hover_name='Country',
    hover_data={'HDI': ':.3f', 'Year': True, 'Country_Code': False},
    animation_frame='Year',
    color_continuous_scale='Viridis',
    range_color=(hdi_map['HDI'].min(), hdi_map['HDI'].max()),
    title='Human Development Index (HDI) по миру: Развитие во времени',
    labels={'HDI': 'HDI Value'},
    projection='natural earth'
)
fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth',
        bgcolor='rgba(0,0,0,0)'
    ),
    width=1000,
    height=800,
    title_font_size=20,
    margin={"r": 0, "t": 80, "l": 0, "b": 0}
)
if len(fig.layout.updatemenus) > 0:
    fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 800
    fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 500
fig.update_coloraxes(
    colorbar_title_text='HDI Value',
    colorbar_title_side='right'
)
fig.show(renderer='jupyterlab')
print("\n=== Статистика по годам ===")
yearly_stats = hdi_map.groupby('Year')['HDI'].agg(['mean', 'min', 'max', 'count']).round(3)
yearly_stats.columns = ['Средний HDI', 'Минимальный HDI', 'Максимальный HDI', 'Количество стран']
display(yearly_stats.tail(10))

Найдено 5940 записей HDI
Страны: 193
Годы: 1990 - 2023

Данные для карты: 5940 записей
Уникальных стран: 193



=== Статистика по годам ===


,Средний HDI,Минимальный HDI,Максимальный HDI,Количество стран
Year,,,,
2014,0.716,0.319,0.956,192
2015,0.720,0.317,0.959,192
2016,0.723,0.297,0.962,192
2017,0.727,0.292,0.965,192
2018,0.731,0.371,0.966,192
2019,0.734,0.279,0.969,192
2020,0.729,0.386,0.969,192
2021,0.729,0.339,0.969,192
2022,0.739,0.385,0.967,192
